# 초구 예측 — model_14 (3단계: 위치에 상황 변수)

- 위치 1단계 보완 끝 (model_11–13), 0장 (README 4-28)
  - 0-1 축별 구분력: 존 안/밖 AUC 0.55 (거의 0), 좌우·높이 0.62 (대부분 구종 덕)
  - 0-2 상한: 퍼짐의 대부분이 실행 오차 -> 노린 위치를 완벽히 알아도 H/S/C 정확도 44–46% 정도
  - 0-3 변수 사전 확인: 상황 +3.9 ×1000 (가장 큼) > 타자 성향 > 나머지 거의 0
- 모델: model_13 위치(24칸 확률)를 **출발점(init_score)**으로 두고 LightGBM이 **상황**에 따른 보정만 학습 (구종 model_05와 같은 구조, 사용자 결정)
  - 상황 변수: 주자 1·2·3루, 1루 빔(득점권인데 1루 비었음), 점수 차, 이닝, 아웃, 홈/원정, 타순 + 구종 원-핫·타자 손
  - 학습 2018 -> 2019로 LightGBM 파라미터·트리 수 선택 / 2020은 같은 값으로 2018–19 다시 학습 (2017은 model_13 위치가 계산 안 됨), 재학습은 시즌 단위 먼저
  - 정규분포(좌표 NLL·거리)는 model_13 그대로, 24칸 확률만 보정
  - 구종(model_09)은 그대로 -> 실전 변화 = 순수한 위치 개선 효과
- 비교 대상: model_13

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_14"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

- 대상: model_13 위치 (README 4-27), 판단 2019 / 2020은 같은 방향인지
- **실제 구종 기준** (위치 모델 자체) 위주, 실전(구종 확률로 섞음)도 같이

#### 0-0. 공통: model_13 위치 재현

- 위치 함수는 model_13 3장 복사, 값도 model_13 -> 실제 구종 기준 24칸 log loss가 결과 파일과 같은지 확인
- 공마다 정규분포 (중심 cm, 분산, 공분산) — 실제 구종 기준 1개 + 구종 9개 각각 (실전 섞기용)

In [2]:
DIAG = "model_13"
pp17 = fc.load_results([DIAG])[DIAG]
LOC_REF, LOC_REF_HL = "decay", 173.0  # model_13 위치 값
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0
LOC_SEASON_F = 0.707
LOC_AC = (None, 12.5, 17.3)
LOC_VAR = (200.0, 9.74, False, 316.0)
ZONE_M = 32
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g


LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
G = {}  # 연도 -> (q, 실제 구종 정규분포 g, 24칸 확률 (n, 24), 구종별 정규분포 목록, 구종 확률 (n, 9), 실전 24칸 확률 (n, 24))
for yr, q in [(fc.VAL_YEAR, q_val), (fc.TEST_YEAR, q_test)]:
    i, yz = np.arange(len(q)), q["zone"].map(fc.ZONE_IDX).to_numpy()
    PZ, g = loc_given_type(q, yr, q["pitch_type"].to_numpy(), **LOC17)
    d17 = fc.align(pp17, fp, yr)
    assert np.allclose(-np.log(np.clip(PZ[i, yz], 1e-12, None)), d17["ll_zone_true"].to_numpy()), f"{yr} model_13 위치 재현 실패"
    GT = [loc_given_type(q, yr, t, **LOC17)[1] for t in CLASSES]  # 구종 9개 각각 가정한 정규분포
    G[yr] = (q, g, PZ, GT, d17[fc.P_TYPE].to_numpy(), d17[fc.P_ZONE].to_numpy())
print("model_13 실제 구종 기준 24칸 log loss (재현 확인):", {yr: round(float(fc.align(pp17, fp, yr)["ll_zone_true"].mean()), 5) for yr in G})

model_13 실제 구종 기준 24칸 log loss (재현 확인): {2019: 2.88724, 2020: 2.88712}


### 0-1. 축별 구분력

- 축 (공마다 0/1, 높이는 그 타자 존 기준)
  - **좌우**: 몸쪽 (x > 0) vs 바깥쪽 / **높이**: 높게 (존 가운데 높이보다 위) vs 낮게 / **존 안·밖**: C(Chase) vs H·S
  - 확률: 좌우·높이는 정규분포에서 바로 (P(x > 0) = Φ(중심 / SD)), 존 안·밖은 24칸 확률의 C 칸 합 / 실전은 구종 확률로 섞음
- 지표
  - log loss와 **기준선 대비 개선률** = 1 − log loss / 기준선 log loss (기준선 = 지난 시즌까지 분석 대상 투수 전체 비율, 모든 공에 같은 확률)
  - **AUC** (0.5 = 구분 못 함, 1 = 완벽), Brier 분해: 구분력(resolution) / 불확실성 = 설명한 몫, 보정 오차
- H/S/C 구분력이 거의 0이었던 것(베이스라인 4-1)이 축마다 같은지 -> 어느 축은 구분하고 어느 축은 못 하는지

In [3]:
from scipy.special import ndtr

def axes_true(q):  # 공마다 축별 실제 값 (0/1) dict
    mid = (q["TOP_SZ_CM"] + q["BOTTOM_SZ_CM"]).to_numpy() / 2  # 존 가운데 높이 (그 타자 존)
    return {"좌우 (몸쪽)": (q["x"].to_numpy() > 0).astype(float),
            "높이 (높게)": (q["CROSS_PLATE_Z_CM"].to_numpy() > mid).astype(float),
            "존 밖 (C)": q["zone_group"].eq("C").to_numpy().astype(float)}

def axes_prob(q, g, PZ):  # 정규분포 g·24칸 확률 PZ -> 축별 확률 dict
    mx, mz, vx, vz, _ = g
    mid = (q["TOP_SZ_CM"] + q["BOTTOM_SZ_CM"]).to_numpy() / 2
    return {"좌우 (몸쪽)": ndtr(mx / np.sqrt(vx)), "높이 (높게)": ndtr((mz - mid) / np.sqrt(vz)), "존 밖 (C)": PZ[:, fc.ZONE_GROUP == "C"].sum(1)}

def axes_prob_mix(q, GT, Pt, Pmix):  # 실전: 구종 확률로 섞은 축별 확률
    out = {k: 0.0 for k in ["좌우 (몸쪽)", "높이 (높게)"]}
    for j, gt in enumerate(GT):
        a = axes_prob(q, gt, np.zeros((len(q), 24)))
        for k in out:
            out[k] = out[k] + Pt[:, j] * a[k]
    out["존 밖 (C)"] = Pmix[:, fc.ZONE_GROUP == "C"].sum(1)
    return out

def bll(p, o):  # 0/1 log loss
    p = np.clip(p, 1e-12, 1 - 1e-12)
    return -(o * np.log(p) + (1 - o) * np.log(1 - p)).mean()

def auc(p, o):  # Mann-Whitney AUC (같은 값은 평균 순위)
    r = pd.Series(p).rank().to_numpy()
    n1, n0 = o.sum(), len(o) - o.sum()
    return (r[o == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def brier_parts(p, o, nb=10):  # Brier 분해 (구간 10개): 보정 오차, 구분력, 불확실성
    b = np.clip((p * nb).astype(int), 0, nb - 1)
    t = pd.DataFrame({"b": b, "p": p, "o": o}).groupby("b").agg(n=("p", "size"), pm=("p", "mean"), om=("o", "mean"))
    obar = o.mean()
    return (t.n * (t.pm - t.om) ** 2).sum() / len(p), (t.n * (t.om - obar) ** 2).sum() / len(p), obar * (1 - obar)

past_rate = {yr: {k: v.mean() for k, v in axes_true(fp[fp["GYEAR"] < yr]).items()} for yr in G}  # 기준선: 지난 시즌까지 비율
rows = []
for yr, (q, g, PZ, GT, Pt, Pmix) in G.items():
    y = axes_true(q)
    for kind, P in [("실제 구종", axes_prob(q, g, PZ)), ("실전", axes_prob_mix(q, GT, Pt, Pmix))]:
        for ax, o in y.items():
            p = P[ax]
            base = bll(np.full(len(o), past_rate[yr][ax]), o)
            rel, res, unc = brier_parts(p, o)
            rows.append({"연도": yr, "기준": kind, "축": ax, "실제 비율": o.mean(), "예측 평균": p.mean(), "log loss": bll(p, o), "기준선 log loss": base,
                         "개선률 %": (1 - bll(p, o) / base) * 100, "AUC": auc(p, o), "구분력/불확실성 %": res / unc * 100, "보정 오차": rel})
AX = pd.DataFrame(rows).set_index(["기준", "축", "연도"]).unstack("연도")
display(AX[["실제 비율", "log loss", "기준선 log loss", "개선률 %", "AUC", "구분력/불확실성 %", "보정 오차"]].round(4))

실제 비율         log loss         기준선 log loss           개선률 %             AUC         구분력/불확실성 %           보정 오차        
연도               2019    2020     2019    2020         2019    2020    2019    2020    2019    2020       2019    2020    2019    2020
기준    축                                                                                                                               
실전    높이 (높게)  0.4967  0.4969   0.6873  0.6890       0.7072  0.6990  2.8088  1.4242  0.5598  0.5524     1.0318  0.6611  0.0003  0.0001
      존 밖 (C)  0.3925  0.3853   0.6689  0.6660       0.6721  0.6683  0.4761  0.3430  0.5295  0.5228     0.3060  0.0557  0.0005  0.0003
      좌우 (몸쪽)  0.4003  0.3799   0.6699  0.6605       0.6732  0.6646  0.4832  0.6237  0.5468  0.5517     0.5676  0.7319  0.0002  0.0001
실제 구종 높이 (높게)  0.4967  0.4969   0.6735  0.6755       0.7072  0.6990  4.7651  3.3561  0.6139  0.6055     3.8145  3.2304  0.0003  0.0003
      존 밖 (C)  0.3925  0.3853   0.6659  0.6640       0.6721  0.6683  0.9099  0.6377  0.5483  0.5428     0.8441  0.4808  0.0006  0.0002
      좌우 (몸쪽)  0.4003  0.3799   0.6472  0.6407       0.6732  0.6646  3.8565  3.6065  0.6252  0.6196     4.7727  4.3408  0.0001  0.0002

#### 🔍 0-1 결과: 좌우·높이는 조금 구분, 존 안/밖은 거의 못 함, 구분력 대부분은 구종에서 옴
| 실제 구종 기준 (2019 / 2020) | AUC | 기준선 대비 개선률 | 구분력/불확실성 |
| --- | --- | --- | --- |
| 좌우 (몸쪽) | 0.625 / 0.620 | 3.9% / 3.6% | 4.8% / 4.3% |
| 높이 (높게) | 0.614 / 0.606 | 4.8% / 3.4% | 3.8% / 3.2% |
| **존 밖 (C)** | **0.548 / 0.543** | **0.9% / 0.6%** | **0.8% / 0.5%** |

- 실전(구종 확률로 섞음)은 좌우 AUC 0.547 / 0.552, 높이 0.560 / 0.552로 크게 떨어짐 -> 지금 위치 구분력의 대부분은 "어떤 구종인가"에서 옴 (변화구는 낮게, 구종마다 좌우 쏠림)
- 존 안/밖은 실제 구종을 알아도 거의 0 -> 베이스라인 때부터의 "H/S/C 구분력 0"은 주로 존 안/밖 축 문제
- 보정 오차는 모두 0.0006 이하 -> 확률은 맞고 구분만 못 함


### 0-2. 상한: 노린 위치를 완벽히 알면

**0-2-1. 실행 오차의 단서 (데이터)**

- 같은 투수·구종·손이라도 공은 흩어짐 = 노린 위치가 공마다 다름(예측 가능할 수도) + 노린 대로 못 던짐(실행 오차, 예측 불가)
- 데이터만으로는 둘을 나눌 수 없음 -> 단서: **3-0 카운트**는 거의 다 존 한가운데를 노림 -> 3-0 퍼짐 ≈ 실행 오차 (+ 작은 노림 차이) = 실행 오차의 위쪽 한계
- 투수×구종×손 안 SD (지난 기록 2017–2019, 구종 의심 제외, 공 수 − 1 가중), 직구만 / 전체

In [4]:
ok = m[~m["label_doubt"] & m["GYEAR"].le(fc.VAL_YEAR)].copy()
ok["cnt"] = ok["ball_before"].astype(int).astype(str) + "-" + ok["strike_before"].astype(int).astype(str)
ok["z_cm"] = ok["CROSS_PLATE_Z_CM"] - (ok["TOP_SZ_CM"] + ok["BOTTOM_SZ_CM"]) / 2  # 존 가운데 기준 높이 cm (타자 키 차이 제거)

def within_sd(d):  # 투수×구종×손 안 SD (좌우 cm, 높이 cm), 공 수 − 1 가중
    g = d.groupby(["pitcher_code", "pitch_type", "same_side"])
    n = g.size()
    w = (n - 1).clip(lower=0)
    vx, vz = g["plate_x_in_CM"].var().fillna(0), g["z_cm"].var().fillna(0)
    return np.sqrt((vx * w).sum() / w.sum()), np.sqrt((vz * w).sum() / w.sum()), int(n.sum())

rows = []
for cnt in ["0-0", "3-0", "3-1", "2-0", "1-1", "0-2", "1-2"]:
    for nm, d in [("직구", ok[ok["pitch_type"].eq("FAST")]), ("전체 구종", ok)]:
        sx, sz, n = within_sd(d[d["cnt"].eq(cnt)])
        rows.append({"카운트": cnt, "구종": nm, "공": n, "좌우 SD cm": sx, "높이 SD cm": sz})
WSD = pd.DataFrame(rows).set_index(["구종", "카운트"])
display(WSD.round(1))
f30 = (WSD.loc[("직구", "3-0"), ["좌우 SD cm", "높이 SD cm"]] / WSD.loc[("직구", "0-0"), ["좌우 SD cm", "높이 SD cm"]])
print("직구 3-0 / 0-0 SD 비율 (실행 오차 몫의 위쪽 한계 단서):", f30.round(2).to_dict())

,,공,좌우 SD cm,높이 SD cm
구종,카운트,,,
직구,0-0,19101,25.5,26.6
전체 구종,0-0,38094,24.5,28.6
직구,3-0,1416,27.5,26.2
전체 구종,3-0,1626,26.9,26.5
직구,3-1,2296,21.8,24.2
전체 구종,3-1,3306,21.6,25.2
직구,2-0,3249,24.8,26.3
전체 구종,2-0,5024,24.0,27.2
직구,1-1,6713,25.4,25.5


직구 3-0 / 0-0 SD 비율 (실행 오차 몫의 위쪽 한계 단서): {'좌우 SD cm': 1.08, '높이 SD cm': 0.99}


**0-2-2. 시뮬레이션**

- 가정: model_13 정규분포 (중심 μ, 공분산 Σ)가 맞다고 봄 (80% 타원 0.81로 보정은 맞음, 4-27)
  - Σ = Σ_노림 + Σ_실행, Σ_실행 = f² · Σ (실행 오차가 전체 퍼짐의 f배 SD), Σ_노림 = (1 − f²) · Σ
  - 노린 위치 T ~ N(μ, Σ_노림), 실제 위치 = T + N(0, Σ_실행)
  - **완벽한 모델** = 노린 위치 T를 정확히 안다 -> N(T, Σ_실행)으로 예측 / **지금 모델** = N(μ, Σ)
- f를 바꿔 가며 (1이면 노림을 알아도 소용없음 = 지금이 상한, 0에 가까울수록 맞히기 쉬움) 24칸 log loss·H/S/C 정확도·축별 AUC 상한
- 2019 초구, 실제 구종 기준, 공마다 8번 뽑음

In [5]:
rng = np.random.default_rng(0)
q, g, PZ, _, _, _ = G[fc.VAL_YEAR]
mx, mz, vx, vz, c = g
K = 8  # 공마다 뽑는 수
idx = np.repeat(np.arange(len(q)), K)
qr = q.iloc[idx].reset_index(drop=True)  # 공 K번 반복 (존 높이 등)
top, bot = qr["TOP_SZ_CM"].to_numpy(), qr["BOTTOM_SZ_CM"].to_numpy()
mid = (top + bot) / 2
sx, sz = np.sqrt(vx)[idx], np.sqrt(vz)[idx]
rho = (c / np.sqrt(vx * vz))[idx]  # 좌우·높이 상관
z1, z2 = rng.standard_normal((len(idx), 2)), rng.standard_normal((len(idx), 2))

def corr_draw(zz, scale):  # 상관 rho, SD (sx, sz)·scale인 2차원 정규 잡음
    ex = zz[:, 0]
    ez = rho * zz[:, 0] + np.sqrt(1 - rho ** 2) * zz[:, 1]
    return ex * sx * scale, ez * sz * scale

def zone_of(x, z):  # 좌표 -> 24칸 번호
    xi = np.digitize(x, fc.X_EDGES)
    zi = (z[:, None] >= fc.z_edges(top, bot)).sum(1)
    return pd.Series(fc.CELL_NAMES[zi * 6 + xi]).map(fc.ZONE_IDX).to_numpy()

def zone_probs_chunk(gg, qq, n=12000):  # 24칸 적분을 나눠서 (메모리)
    return np.vstack([fc.zone_from_gauss_cov(tuple(a[s:s + n] for a in gg), qq.iloc[s:s + n], ZONE_M) for s in range(0, len(qq), n)])

grp = pd.Series(fc.ZONE_GROUP).map({"H": 0, "S": 1, "C": 2}).to_numpy()
rows = []
PZr = PZ[idx]  # 지금 모델 24칸 확률 (공마다 같음)
for f in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.3]:
    ax_, az_ = corr_draw(z1, np.sqrt(1 - f ** 2))  # 노림 = μ + 노림 잡음
    Tx, Tz = mx[idx] + ax_, mz[idx] + az_
    ex_, ez_ = corr_draw(z2, f)  # 실행 오차
    Yx, Yz = Tx + ex_, Tz + ez_
    yz = zone_of(Yx, Yz)
    if f < 1.0:
        Po = zone_probs_chunk((Tx, Tz, (f * sx) ** 2, (f * sz) ** 2, rho * f * sx * f * sz), qr)  # 완벽한 모델 24칸 확률
        ox, oz = ndtr(Tx / (f * sx)), ndtr((Tz - mid) / (f * sz))
    else:
        Po, ox, oz = PZr, ndtr(mx[idx] / sx), ndtr((mz[idx] - mid) / sz)
    i = np.arange(len(idx))
    yx_, yzz_, yc_ = (Yx > 0).astype(float), (Yz > mid).astype(float), (grp[yz] == 2).astype(float)
    rows.append({"f (실행 SD / 전체 SD)": f, "실행 SD 좌우·높이 cm": f"{(f * sx).mean():.1f} · {(f * sz).mean():.1f}",
                 "24칸 log loss 지금 모델": -np.log(np.clip(PZr[i, yz], 1e-12, None)).mean(),
                 "24칸 log loss 완벽한 모델": -np.log(np.clip(Po[i, yz], 1e-12, None)).mean(),
                 "H/S/C 정확도 지금": ((PZr @ fc.GROUP2).argmax(1) == grp[yz]).mean(), "H/S/C 정확도 완벽": ((Po @ fc.GROUP2).argmax(1) == grp[yz]).mean(),
                 "AUC 좌우 완벽": auc(ox, yx_), "AUC 높이 완벽": auc(oz, yzz_), "AUC 존 밖 완벽": auc(Po[:, fc.ZONE_GROUP == "C"].sum(1), yc_)})
SIM = pd.DataFrame(rows).set_index("f (실행 SD / 전체 SD)")
display(SIM.round(4))
real = fc.align(pp17, fp, fc.VAL_YEAR)["ll_zone_true"].mean()
print(f"실제 2019 24칸 log loss (실제 구종) {real:.4f} vs 시뮬레이션 지금 모델 {SIM.iloc[0]['24칸 log loss 지금 모델']:.4f} (보정이 맞으면 비슷)")
base24 = fp.loc[fp["GYEAR"] < fc.VAL_YEAR, ZONES].mean().to_numpy()  # 기준선: 지난 시즌까지 24칸 비율
print(f"기준선 (모든 공에 지난 시즌 24칸 비율) 2019 log loss {-np.log(base24[q['zone'].map(fc.ZONE_IDX).to_numpy()]).mean():.4f}")

,실행 SD 좌우·높이 cm,24칸 log loss 지금 모델,24칸 log loss 완벽한 모델,H/S/C 정확도 지금,H/S/C 정확도 완벽,AUC 좌우 완벽,AUC 높이 완벽,AUC 존 밖 완벽
f (실행 SD / 전체 SD),,,,,,,,
1.0,23.8 · 28.0,2.8741,2.8741,0.4224,0.4224,0.6252,0.6247,0.5647
0.9,21.4 · 25.2,2.8752,2.6970,0.4223,0.4432,0.7352,0.7290,0.6200
0.8,19.0 · 22.4,2.8747,2.5027,0.4243,0.4753,0.8027,0.7983,0.6828
0.7,16.7 · 19.6,2.8753,2.2954,0.4234,0.5092,0.8539,0.8507,0.7448
0.6,14.3 · 16.8,2.8750,2.0641,0.4230,0.5524,0.8951,0.8928,0.8031
0.5,11.9 · 14.0,2.8767,1.8062,0.4231,0.6059,0.9277,0.9258,0.8580
0.3,7.1 · 8.4,2.8771,1.1886,0.4225,0.7389,0.9742,0.9738,0.9442


실제 2019 24칸 log loss (실제 구종) 2.8872 vs 시뮬레이션 지금 모델 2.8741 (보정이 맞으면 비슷)
기준선 (모든 공에 지난 시즌 24칸 비율) 2019 log loss 3.0102


#### 🔍 0-2 결과: 실행 오차가 퍼짐의 대부분이라 상한도 높지 않음
- 실행 오차 단서: 직구 3-0 퍼짐(좌우 27.5 · 높이 26.2cm)이 0-0(25.5 · 26.6cm)과 같음 -> 존 가운데를 노릴 때도 이만큼 흩어짐
  - 3-0은 일부러 빼는 공도 섞여 있을 수 있음 -> 3-1(21.8 · 24.2cm)로 보면 0-0의 0.85 · 0.91배
  - -> 실행 오차가 퍼짐의 f ≈ 0.85–1.0 (SD 기준) 정도로 봄 (데이터만으로 정확히는 못 나눔)
- 시뮬레이션 (2019, 실제 구종 기준, 노린 위치를 완벽히 아는 모델)

| f | 24칸 log loss | H/S/C 정확도 | AUC 좌우 · 높이 · 존 밖 |
| --- | --- | --- | --- |
| 1.0 (= 지금 모델) | 2.874 | 42.2% | 0.625 · 0.625 · 0.565 |
| 0.9 | 2.697 | 44.3% | 0.735 · 0.729 · 0.620 |
| 0.8 | 2.503 | 47.5% | 0.803 · 0.798 · 0.683 |
| 0.5 | 1.806 | 60.6% | 0.928 · 0.926 · 0.858 |

- 지금 모델은 기준선(지난 시즌 24칸 비율 3.010)보다 0.12 좋아짐 / f 0.85–0.9면 노린 위치를 **완벽히** 알아도 0.18–0.27 더 좋아지는 정도, H/S/C 정확도는 44–46%까지
  - 실제 변수는 노린 위치의 일부만 알려 주므로 현실적인 개선은 이보다 훨씬 작음
- 시뮬레이션 지금 모델 2.874 vs 실제 2.887 -> 정규분포 가정과 보정이 대체로 맞음 (차이 0.013)
- 남은 여지 (지금 -> f 0.9 상한, AUC): 좌우 0.625 -> 0.735, 높이 0.614 -> 0.729, 존 밖 0.548 -> 0.620

#### 🔍 0장 결론
- 위치는 실행 오차가 커서 노린 위치를 알아도 크게는 못 맞힘 -> 기대치를 낮게 잡아야 함
- 그래도 구분력이 0은 아님: 좌우·높이는 구종 덕에 AUC 0.62, 존 안/밖은 거의 0이지만 상황 신호가 이미 보임 (득점권 +0.036 · 주자 없음 −0.043, 4-9·4-23)
- -> 상황 변수는 시험해 볼 만하지만 큰 개선은 기대하기 어려움


### 0-3. 위치 변수 사전 확인 (2단계 4-7과 같은 방식)

- 지금 위치 확률(model_13, 실제 구종 기준 24칸)의 log를 **출발점(init_score)**으로 두고 LightGBM이 변수에 따른 보정만 학습 (구종 model_05와 같은 구조)
  - 학습 2018 -> 확인 2019 (2017은 지난 시즌 기록이 없어 model_13 위치가 계산되지 않음 -> 학습에서 뺌), 2020 안 씀
  - 구조 대조 = 구종 원-핫 + 타자 손만 넣은 LightGBM (보정 구조 자체의 효과를 빼려고) -> 묶음마다 이 위에 더해서 이득
  - 이득 ×1000 = 2019 24칸 log loss 감소 (구조 대조 대비), 경기 단위 95% 구간 + 존 밖(C) AUC
  - 하나씩 더하기 / 전부 / 전부에서 하나씩 빼기 (고유 몫)
- 변수 묶음: README 5장 3번 목록 1–7 + **목록에 없지만 실험에서 위치 신호가 보였던 것** (사용자 요청)

| 묶음 | 변수 | 근거 |
| --- | --- | --- |
| 1 오늘 제구 | 그 경기 앞 공 볼 비율, 앞 타석 초구 스트라이크 비율 | 목록 1 |
| 2 타자 성향 | 초구 스윙률, 존 밖·존 안 공 스윙률, 타석당 장타 비율 (전날까지) | 목록 2 |
| 3 이 타자와 앞 대결 | 같은 타자 몇 번째, 그 경기 바로 앞 대결 결과 (안타·장타·볼넷·삼진·아웃) | 목록 3 |
| 4 상황 | 주자 1·2·3루, 1루 빔(득점권인데 1루 비었음), 점수 차, 이닝, 아웃, 홈/원정, 타순 | 목록 4, 4-9·4-23 득점권 C +0.036 |
| 5 포수·구장·월 | 포수·구장별 H/S/C 잔차 (전날까지), 월 | 목록 5 |
| 6 투수 상태 | 그 경기 투구 수, 선발/불펜, 등판 간격, 역할 전환 | 목록 6 |
| 7 투수×상황 반응 | 투수×상황 칸(첫 타자·득점권·재대결·그 밖)별 H/S/C 잔차 | 목록 7 |
| 추가: 시즌 등판 순번 | 그 시즌 몇 번째 등판 | 4-23·4-24: 시즌 첫 등판 C 과다 예측(rC −0.07)이 시즌 초 배수로 안 고쳐짐 |
| 추가: 직전 타자 결과 | 바로 앞 타석(다른 타자) 초구 결과·타석 결과 | 구종 model_07 채택 변수, 위치 목록엔 없음 |
| 추가: 리그 최근 위치 잔차 | 분석 대상 투수 전체 최근(반감 500구) H/S/C 잔차 | 4-23: 2019 높이·C 추세 (리그 추세), 구종 4-20 방식 |

- 새 변수 함수 (fp_common, 미래 정보 검사): `batter_location_features`, `same_batter_features`, `appearance_features`, `group_residual_loc`

In [6]:
# 0-3-1. 출발점: model_13 위치 (실제 구종 기준) 2018–2019 / 변수 표
PZ_ALL = np.full((len(fp), len(ZONES)), np.nan)
for yr in (2018, fc.VAL_YEAR):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_ALL[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
assert np.allclose(-np.log(np.clip(PZ_ALL[val][np.arange(val.sum()), y_zone[val]], 1e-12, None)), fc.align(pp17, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy())
ok18 = fp["GYEAR"].eq(2018).to_numpy() & np.isfinite(PZ_ALL).all(1)
print(f"2018 초구 {fp['GYEAR'].eq(2018).sum():,}개 중 위치 계산됨 {ok18.sum():,}개")

for fn in (fc.batter_location_features, fc.same_batter_features, fc.appearance_features):  # 새 변수 미래 정보 검사
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(2018)], n=150)
    assert bad.empty, bad

SIT = fc.situation_features(m, fp)
FLOW = fc.game_flow_features(m, fp)
HIST = fc.pitcher_history_features(m, fp)[["rest", "role_change"]]
OPP = fc.opponent_features(m, fp)[["home", "bat_swing"]]  # bat_n은 해마다 범위가 커져 안 씀 (4-12)
BAT = fc.batter_location_features(m, fp)
SB = fc.same_batter_features(m, fp)
APP = fc.appearance_features(m, fp)
P3 = PZ_ALL @ fc.GROUP2  # H/S/C 확률 (2017·2020은 빈 값 -> 잔차에서 안 셈)
risp = (SIT["base2"] + SIT["base3"]) > 0
fpx = fp.assign(sit_cell=np.select([SIT["pc"].eq(0), risp, SIT["tto"].ge(2)], ["첫타자", "득점권", "재대결"], "그밖"), _all=1)
RES = pd.concat([fc.group_residual_loc(fpx, P3, ["catcher_code"], fpx, prefix="catcher"),
                 fc.group_residual_loc(fpx, P3, ["VENUE_ID"], fpx, prefix="venue"),
                 fc.group_residual_loc(fpx, P3, ["pitcher_code", "sit_cell"], fpx, prefix="psit"),
                 fc.group_residual_loc(fpx, P3, ["_all"], fpx, half_life=500.0, prefix="league")], axis=1)
XL = pd.concat([fp[CLASSES + ["same_side"]], SIT, FLOW, HIST, OPP, BAT, SB, APP, RES], axis=1)
XL["first_open"] = (risp & SIT["base1"].eq(0)).astype(int)  # 득점권인데 1루 비었음 (거르기 가능)
XL["month"] = fp["game_date"].dt.month.clip(4, 9)
hsc = lambda pre: [f"{pre}_{c}" for c in fc.HSC]
BASE_COLS = CLASSES + ["same_side"]  # 구조 대조
BUNDLES = {
    "1 오늘 제구": ["ball_rate", "fs_rate"],
    "2 타자 성향": ["bat_swing", "bat_chase", "bat_zswing", "bat_xbh"],
    "3 이 타자와 앞 대결": ["tto"] + ["sb_prev_" + c for c in fc.SB_CATS],
    "4 상황": ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"],
    "5 포수·구장·월": hsc("catcher") + hsc("venue") + ["month"],
    "6 투수 상태": ["pc", "starter", "rest", "role_change"],
    "7 투수×상황 반응": hsc("psit"),
    "추가: 시즌 등판 순번": ["app_no"],
    "추가: 직전 타자 결과": ["prev_first_" + c for c in fc.FIRST_CATS] + ["prev_how_" + c for c in fc.HOW_CATS],
    "추가: 리그 최근 위치 잔차": hsc("league"),
}
display(XL.loc[ok18 | val, sum(BUNDLES.values(), [])].describe().T[["count", "mean", "min", "max"]].round(3))

2018 초구 12,395개 중 위치 계산됨 12,395개
미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,count,mean,min,max
ball_rate,23367.0,0.352,0.000,1.000
fs_rate,23365.0,0.611,0.000,1.000
bat_swing,25853.0,0.299,0.124,0.481
bat_chase,25853.0,0.227,0.128,0.373
bat_zswing,25853.0,0.655,0.531,0.811
bat_xbh,25853.0,0.079,0.021,0.177
tto,25853.0,1.586,1.000,5.000
sb_prev_없음,25853.0,0.581,0.000,1.000
sb_prev_안타,25853.0,0.075,0.000,1.000
sb_prev_장타,25853.0,0.030,0.000,1.000


In [7]:
# 0-3-2. 묶음별 이득 (학습 2018 -> 2019)
from scipy.special import log_softmax
NZ = len(ZONES)
yz_all = y_zone
INITZ = np.log(np.clip(PZ_ALL, 1e-6, None))
INITZ -= np.nanmean(INITZ, 1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
itr = np.flatnonzero(ok18)
i19 = np.flatnonzero(val)
gid_all = fp["gameID"].to_numpy()
LGBZ = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, num_leaves=7, min_data_in_leaf=50, lambda_l2=1000.0,
            feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)
g18 = np.unique(gid_all[itr])
es = np.isin(gid_all[itr], np.random.default_rng(1).choice(g18, len(g18) // 5, replace=False))  # 조기 종료용 경기 20%
t_, v_ = itr[~es], itr[es]

def screen_p(cols):  # cols로 학습 -> 2019 24칸 확률 (n, 24), 트리 수
    dtr = lgb.Dataset(XL[cols].iloc[t_], yz_all[t_], init_score=INITZ[t_], free_raw_data=False)
    dva = lgb.Dataset(XL[cols].iloc[v_], yz_all[v_], init_score=INITZ[v_], reference=dtr)
    bst = lgb.train(LGBZ, dtr, 3000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    raw = bst.predict(XL[cols].iloc[i19], raw_score=True, num_iteration=bst.best_iteration) + INITZ[i19]
    return np.exp(log_softmax(raw, axis=1)), bst.best_iteration

def game_ci19(d, B=1000):  # 경기 단위 부트스트랩 95% 구간 (2019)
    s = pd.DataFrame({"g": gid_all[i19], "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

y19 = yz_all[i19]
oc19 = (fc.ZONE_GROUP[y19] == "C").astype(float)
nll19 = lambda P: -np.log(np.clip(P[np.arange(len(i19)), y19], 1e-12, None))
P_base, n_base = screen_p(BASE_COLS)
ll_base = nll19(P_base)
ll17 = nll19(PZ_ALL[i19])
rows = [{"비교": "model_13 (트리 없음)", "트리 수": 0, "2019 24칸 ll": ll17.mean(), "이득 ×1000": (ll_base - ll17).mean() * 1000,
         "존 밖 AUC": auc(PZ_ALL[i19][:, fc.ZONE_GROUP == "C"].sum(1), oc19)},
        {"비교": "구조 대조 (구종 + 손)", "트리 수": n_base, "2019 24칸 ll": ll_base.mean(), "이득 ×1000": 0.0,
         "존 밖 AUC": auc(P_base[:, fc.ZONE_GROUP == "C"].sum(1), oc19)}]
ALL = sum(BUNDLES.values(), [])
runs = [("더함", nm, BASE_COLS + cols) for nm, cols in BUNDLES.items()] + [("전부", "전부", BASE_COLS + ALL)] + \
       [("전부에서 뺌", nm, BASE_COLS + [c for c in ALL if c not in cols]) for nm, cols in BUNDLES.items()]
LLS = {}
for kind, nm, cols in runs:
    P, n_ = screen_p(cols)
    LLS[(kind, nm)] = nll19(P)
    rows.append({"비교": f"{kind}: {nm}", "트리 수": n_, "2019 24칸 ll": LLS[(kind, nm)].mean(), "이득 ×1000": (ll_base - LLS[(kind, nm)]).mean() * 1000,
                 "존 밖 AUC": auc(P[:, fc.ZONE_GROUP == "C"].sum(1), oc19)})
ll_all = LLS[("전부", "전부")]
for r in rows:  # 95% 구간: 더함·전부 = 구조 대조 대비, 전부에서 뺌 = 전부 대비 (잃는 몫, 양수 = 그 묶음이 필요)
    k = r["비교"]
    if k.startswith("더함") or k.startswith("전부:"):
        d = ll_base - LLS[tuple(k.split(": ", 1))]
    elif k.startswith("전부에서 뺌"):
        d = LLS[tuple(k.split(": ", 1))] - ll_all
        r["이득 ×1000"] = d.mean() * 1000
    else:
        continue
    lo, hi = game_ci19(d)
    r["95% 하한"], r["95% 상한"] = lo * 1000, hi * 1000
SCR = pd.DataFrame(rows).set_index("비교")
display(SCR.round(4))

,트리 수,2019 24칸 ll,이득 ×1000,존 밖 AUC,95% 하한,95% 상한
비교,,,,,,
model_13 (트리 없음),0,2.8872,-0.0933,0.5483,NaN,NaN
구조 대조 (구종 + 손),46,2.8872,0.0000,0.5488,NaN,NaN
더함: 1 오늘 제구,72,2.8877,-0.5802,0.5486,-1.2889,0.0653
더함: 2 타자 성향,118,2.8860,1.1643,0.5485,-0.0793,2.4879
더함: 3 이 타자와 앞 대결,31,2.8872,-0.0467,0.5486,-0.3157,0.2023
더함: 4 상황,206,2.8832,3.9053,0.5557,2.1979,5.6057
더함: 5 포수·구장·월,30,2.8866,0.5950,0.5488,0.1023,1.0951
더함: 6 투수 상태,28,2.8872,-0.0868,0.5485,-0.4447,0.2659
더함: 7 투수×상황 반응,18,2.8869,0.2337,0.5488,-0.0868,0.5706


#### 🔍 0-3 결과: 상황이 가장 큼, 타자 성향이 그다음, 나머지는 거의 0
| 묶음 (이득 ×1000, 2019) | 하나만 더함 (95%) | 전부에서 빼면 잃는 몫 (95%) |
| --- | --- | --- |
| **4 상황** | **+3.91** (+2.20 ~ +5.61) | **+1.67** (+1.15 ~ +2.20) |
| **2 타자 성향** | +1.16 (−0.08 ~ +2.49) | **+1.12** (+0.38 ~ +1.91) |
| 5 포수·구장·월 | +0.60 (+0.10 ~ +1.10) | +0.28 (−0.44 ~ +0.96) |
| 추가: 직전 타자 결과 | −0.25 (−1.13 ~ +0.61) | +0.71 (+0.22 ~ +1.19) |
| 7 투수×상황 반응 | +0.23 (−0.09 ~ +0.57) | +0.32 (−0.27 ~ +0.92) |
| 추가: 리그 최근 위치 잔차 | +0.09 (−0.26 ~ +0.43) | +0.40 (−0.27 ~ +1.07) |
| 추가: 시즌 등판 순번 | −0.01 (−0.28 ~ +0.27) | +0.21 (−0.20 ~ +0.64) |
| 3 이 타자와 앞 대결 | −0.05 (−0.32 ~ +0.20) | +0.02 (−0.18 ~ +0.23) |
| 6 투수 상태 | −0.09 (−0.44 ~ +0.27) | −0.03 (−0.53 ~ +0.44) |
| 1 오늘 제구 | −0.58 (−1.29 ~ +0.07) | −0.29 (−0.84 ~ +0.23) |
| 전부 | +2.42 (+1.11 ~ +3.73) | |

- 구조 대조(구종 + 손만)는 model_13과 같음 (−0.09) -> 보정 구조 자체로는 얻는 것 없음
- **상황**이 유일하게 뚜렷: 존 밖 AUC 0.549 -> 0.556 (4-9·4-23의 득점권·주자 없음 신호와 맞음), 2단계 구종 A(28.5)보다는 훨씬 작음 (0장 상한과 맞음)
- **전부(+2.4) < 상황만(+3.9)**: 학습이 2018 한 해(약 1.2만 개)뿐이라 변수가 많으면 우연한 패턴을 배움 -> 묶음을 하나씩 쌓아야 함
- 타자 성향은 하나만으로는 경계, 전부에서 빼면 뚜렷 -> 상황 다음 후보
- 직전 타자 결과는 하나만은 −, 전부에서 빼면 + (다른 묶음과 함께일 때만) -> 우선순위 낮음
- 오늘 제구·투수 상태·이 타자와 앞 대결·시즌 등판 순번은 효과 없음 (시즌 첫 등판 C 과다 예측도 등판 순번으로는 안 잡힘)
- -> 넣는 순서 제안: **상황 -> 타자 성향 -> (포수·구장·월, 직전 타자 결과는 그 위에서 다시 확인)**


## 모델

In [8]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 구종은 model_09 값, 위치 정규분포는 model_13 값 고정
- 위치 보정 LightGBM: 상황 변수 `LOC_FEAT`, 파라미터는 3-2에서 고름

In [9]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 정규분포: model_13 값 (고정)
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 0.707  # 시즌 초 배수
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k)
LOC_VAR = (200.0, 9.74, False, 316.0)  # 분산 따로 (다른 카운트 퍼짐 반감, k_var, unc, 투수 안 분산 당김 k0)
# ② 위치 상황 보정 (model_14): 24칸 LightGBM, 출발점 = model_13 위치 log 확률
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # 상황 변수 (0-3 묶음 4)
LOC_FEAT = CLASSES + ["same_side"] + LOC_SIT  # 구종 원-핫·타자 손 + 상황
LOC_TRAIN_FIRST = 2018  # 위치 보정 학습 첫 해 (2017은 model_13 위치 없음)
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [10]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m13 = pd.read_parquet("results/model_13/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [11]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [12]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_14    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_14    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: model_13 정규분포 + 상황 보정

### 3-1. model_13 위치 (그대로)

- 위치 함수는 model_13 3장 복사 -> 실제 구종 기준 24칸 log loss가 model_13 결과 파일과 같은지 확인

In [14]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g


LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)  # 실제 구종 기준 model_13 24칸 확률 (2018–2020)
for yr in range(LOC_TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    ll_ = -np.log(np.clip(PZ_TRUE[msk][np.arange(msk.sum()), y_zone[msk]], 1e-12, None))
    assert np.allclose(ll_, fc.align(m13, fp, yr)["ll_zone_true"].to_numpy()), f"{yr} model_13 위치 재현 실패"
print("실제 구종 기준 24칸 = model_13 (재현 확인), 2018 계산된 공:", int(np.isfinite(PZ_TRUE[fp["GYEAR"].eq(2018).to_numpy()]).all(1).sum()))

실제 구종 기준 24칸 = model_13 (재현 확인), 2018 계산된 공: 12395


### 3-2. 상황 보정 LightGBM (2019로 고르기)

- 출발점 = log(model_13 24칸 확률) (공마다 상수는 빼도 같음) -> 상황 정보가 쓸모없으면 model_13 그대로
- 변수: `fc.situation_features` (주자·이닝·아웃·타순·점수 차) + `fc.opponent_features`의 home + 1루 빔, 미래 정보 검사 다시 통과 확인
- 학습 2018 -> 2019 조기 종료로 트리 수, 잎 수 · 잎당 최소 공 · lambda_l2 격자 (범위 끝이면 넓힘), 2019 1등과 0.0002 안이면 덜 극단적인 값 (잎 적게 -> 트리 적게)
- learning_rate 0.05 (24칸이라 0.03은 느림, 0장 0-3과 같음)

In [15]:
def grid_search(score, grid, by="logloss"):  # grid의 모든 조합을 score로 채점 (지표 낮은 순)
    names_ = list(grid)
    combos = pd.MultiIndex.from_product(grid.values(), names=names_).to_frame(index=False)  # 후보의 모든 조합 (한 행 = 한 조합)
    rows = [{**r, **score(**r)} for r in combos.astype(object).where(combos.notna(), None).to_dict("records")]  # NaN -> None으로 되돌려 넘김
    return pd.DataFrame(rows).sort_values(by).reset_index(drop=True)

def finer(cands, best, n=3):
    # 거친 후보 목록에서 best와 양옆 후보 사이를 각각 로그 간격으로 n개씩 나눈 세밀한 후보 (best 가운데, 2n−1개)
    # best가 범위 끝이면 바깥으로 2배 넓힘, best가 None이면 None 그대로
    if best is None:
        return [None]
    nums = sorted(c for c in cands if c is not None)  # None 뺀 숫자 후보
    i = nums.index(best)
    lo = nums[i - 1] if i > 0 else best / 2  # 왼쪽 이웃 (없으면 절반)
    hi = nums[i + 1] if i < len(nums) - 1 else best * 2  # 오른쪽 이웃 (없으면 2배)
    vals = np.r_[np.geomspace(lo, best, n), np.geomspace(best, hi, n)[1:]]  # lo~best, best~hi를 같은 비율 간격으로
    return sorted({float(f"{v:.3g}") for v in vals})  # 유효숫자 3자리로 반올림, 중복 제거

def best_of(table, grid):  # 표 1등 행의 파라미터 dict (NaN -> None, 숫자는 float)
    r = table.iloc[0]
    return {k: (None if pd.isna(r[k]) else float(r[k])) for k in grid}

def marginal(table, grid, by="logloss"):  # 파라미터마다 값별 가장 좋은 by 지표 (나머지는 그 값에서 가장 좋은 조합) -> 어느 파라미터가 민감한지
    out = {}
    for k in grid:
        s = table.assign(**{k: table[k].astype(object).where(table[k].notna(), "None")}).groupby(k, sort=False)[by].min()
        out[k] = s.reindex([("None" if v is None else v) for v in grid[k] if ("None" if v is None else v) in s.index])
    return out

def game_ci(d, gid, B=1000):  # 경기 단위 부트스트랩 95% 구간 (같은 경기 공은 독립이 아님)
    s = pd.DataFrame({"g": gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])




from scipy.special import log_softmax
for fn in (fc.situation_features, fc.opponent_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(LOC_TRAIN_FIRST)], n=150)
    assert bad.empty, bad
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]]], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)  # 득점권인데 1루 비었음
XLOC = XLOC[LOC_FEAT]
NZ = len(ZONES)
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)  # 출발점
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)

def fit_loc(years, lp, rounds=None, val_year=None):  # years 해 초구로 위치 보정 학습 (rounds 없으면 val_year로 조기 종료)
    tr = np.flatnonzero(np.isin(yr_l, years))
    dtr = lgb.Dataset(XLOC.iloc[tr], y_zone[tr], init_score=INIT_L[tr], free_raw_data=False)
    if rounds is None:
        va = np.flatnonzero(yr_l == val_year)
        dva = lgb.Dataset(XLOC.iloc[va], y_zone[va], init_score=INIT_L[va], reference=dtr)
        return lgb.train({**BASE_L, **lp}, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    return lgb.train({**BASE_L, **lp}, dtr, rounds)

iv_l = np.flatnonzero(val)
def score_lgbz(num_leaves, min_data_in_leaf, lambda_l2):  # 한 조합의 2019 24칸 log loss (실제 구종), 트리 수
    lp = {"num_leaves": int(num_leaves), "min_data_in_leaf": int(min_data_in_leaf), "lambda_l2": float(lambda_l2)}
    bst = fit_loc([2018], lp, val_year=fc.VAL_YEAR)
    raw = bst.predict(XLOC.iloc[iv_l], raw_score=True, num_iteration=bst.best_iteration) + INIT_L[iv_l]
    return {"트리 수": bst.best_iteration, "logloss": -log_softmax(raw, axis=1)[np.arange(len(iv_l)), y_zone[iv_l]].mean()}

GRID_LZ = {"num_leaves": [7, 15], "min_data_in_leaf": [20, 50, 200], "lambda_l2": [300.0, 1000.0, 3000.0]}
res_lz = grid_search(score_lgbz, GRID_LZ)
tried_lz = {k: list(v) for k, v in GRID_LZ.items()}
for rnd in range(3):  # 1등이 범위 끝이면 넓힘
    b = res_lz.iloc[0]
    add = {}
    if b.lambda_l2 == max(tried_lz["lambda_l2"]):
        add["lambda_l2"] = [b.lambda_l2 * 3]
    elif b.lambda_l2 == min(tried_lz["lambda_l2"]):
        add["lambda_l2"] = [b.lambda_l2 / 3]
    if b.num_leaves == max(tried_lz["num_leaves"]):
        add["num_leaves"] = [int(b.num_leaves * 2)]
    elif b.num_leaves == min(tried_lz["num_leaves"]):
        add["num_leaves"] = [3]
    if b.min_data_in_leaf == max(tried_lz["min_data_in_leaf"]):
        add["min_data_in_leaf"] = [int(b.min_data_in_leaf * 4)]
    add = {k: v for k, v in add.items() if v[0] not in tried_lz[k]}
    if not add:
        break
    print(f"{rnd + 1}번째 넓히기:", add)
    for k, v in add.items():
        tried_lz[k] += v
        res_lz = pd.concat([res_lz, grid_search(score_lgbz, {**{kk: [b[kk]] for kk in GRID_LZ}, k: v})])
    res_lz = res_lz.drop_duplicates(list(GRID_LZ)).sort_values("logloss").reset_index(drop=True)
TOL = 2e-4
ok_ = res_lz[res_lz["logloss"] <= res_lz["logloss"].iloc[0] + TOL]
pick = ok_.sort_values(["num_leaves", "트리 수"]).iloc[0]  # 평평하면 덜 극단적인 값: 잎 적게 -> 트리 적게
LOC_LGB = {"num_leaves": int(pick.num_leaves), "min_data_in_leaf": int(pick.min_data_in_leaf), "lambda_l2": float(pick.lambda_l2)}
LOC_ROUNDS = int(pick["트리 수"])
ll17_19 = -np.log(np.clip(PZ_TRUE[iv_l][np.arange(len(iv_l)), y_zone[iv_l]], 1e-12, None)).mean()
print(f"2019 24칸 log loss (실제 구종): model_13 {ll17_19:.5f} / 1등 {res_lz.logloss.iloc[0]:.5f} / 선택 {pick.logloss:.5f} {LOC_LGB}, 트리 {LOC_ROUNDS}개")
display(res_lz.head(10).round(5))
for k, s in marginal(res_lz, {k: sorted(v) for k, v in tried_lz.items()}).items():
    print(k, s.round(5).to_dict())

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개
1번째 넓히기: {'lambda_l2': [np.float64(9000.0)], 'num_leaves': [30]}
2019 24칸 log loss (실제 구종): model_13 2.88724 / 1등 2.88334 / 선택 2.88354 {'num_leaves': 7, 'min_data_in_leaf': 20, 'lambda_l2': 1000.0}, 트리 168개


,num_leaves,min_data_in_leaf,lambda_l2,트리 수,logloss
0,15.0,20.0,3000.0,359,2.88334
1,30.0,20.0,3000.0,359,2.88335
2,7.0,20.0,3000.0,359,2.88337
3,7.0,50.0,3000.0,361,2.88339
4,7.0,200.0,3000.0,417,2.88339
5,15.0,50.0,3000.0,361,2.88341
6,15.0,200.0,3000.0,417,2.88343
7,15.0,20.0,9000.0,1032,2.88346
8,15.0,20.0,1000.0,168,2.88351
9,7.0,20.0,1000.0,168,2.88354


num_leaves {7: 2.88337, 15: 2.88334, 30: 2.88335}
min_data_in_leaf {20: 2.88334, 50: 2.88339, 200: 2.88339}
lambda_l2 {300.0: 2.88374, 1000.0: 2.88351, 3000.0: 2.88334, 9000.0: 2.88346}


#### 선택: 잎 7 · 잎당 최소 20 · lambda_l2 1000 · 트리 168개 (평평한 구간 규칙)
- 2019 24칸 log loss (실제 구종): model_13 2.88724 -> 1등 2.88334 (잎 15 · λ 3000 · 트리 359) / 선택 2.88354 (1등과 0.0002 안)
- 규칙대로 1등과 0.0002 안에서 잎 적게 -> 트리 적게 (구종 model_05와 같은 규칙) -> λ 1000 · 트리 168
- λ 범위 끝(3000)이라 9000까지 넓힘 -> 9000은 나쁨 (2.88346), 잎 30도 같음 -> 1등은 안쪽
- 민감도: λ가 가장 큼 (300 -> 3000: 2.88374 -> 2.88334), 잎 수·잎당 최소 공은 거의 평평 -> 구종처럼 강한 규제가 중요

### 3-3. 최종 학습과 진단

- 2019 = 2018 학습, 2020 = 2018–19 학습 (파라미터·트리 수는 3-2 값)
- 진단 (실제 구종 기준): 주자 상황·아웃별 존 밖(C) 실제 − 예측 (4-9·4-23에서 본 치우침이 줄었는지), 존 밖 AUC, 변수 중요도

In [16]:
BST_L = {fc.VAL_YEAR: fit_loc([2018], LOC_LGB, LOC_ROUNDS), fc.TEST_YEAR: fit_loc([2018, 2019], LOC_LGB, LOC_ROUNDS)}

def loc_correct(P, q, year, t=None):  # model_13 24칸 확률 P (n, 24) -> 상황 보정 확률, t = 가정한 구종 (None이면 q의 실제 구종)
    X = XLOC.loc[q.index].copy()
    if t is not None:
        X.loc[:, CLASSES] = np.tile((np.array(CLASSES) == t).astype(float), (len(X), 1))  # 구종 원-핫을 t로
    init = np.log(np.clip(P, 1e-6, None))
    init -= init.mean(1, keepdims=True)
    return np.exp(log_softmax(BST_L[year].predict(X, raw_score=True) + init, axis=1))

def runner3(q):
    return np.select([(q["base2"] + q["base3"]) > 0, q["base1"] > 0], ["득점권", "1루만"], "없음")

rows, imp = [], None
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz = np.arange(len(q)), y_zone[msk]
    P17 = PZ_TRUE[msk]
    P18 = loc_correct(P17, q, yr)
    oc = (fc.ZONE_GROUP[yz] == "C").astype(float)
    d = pd.DataFrame({"주자": runner3(SIT_L.loc[q.index]), "아웃": SIT_L.loc[q.index, "out_cn"].clip(upper=2).to_numpy(), "C 실제": oc,
                      "C 17": P17[:, fc.ZONE_GROUP == "C"].sum(1), "C 18": P18[:, fc.ZONE_GROUP == "C"].sum(1),
                      "이득": (np.log(np.clip(P18[i, yz], 1e-12, None)) - np.log(np.clip(P17[i, yz], 1e-12, None))) * 1000})
    for var in ["주자", "아웃"]:
        for v, g in d.groupby(var):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "rC model_13": (g["C 실제"] - g["C 17"]).mean(),
                         "rC model_14": (g["C 실제"] - g["C 18"]).mean(), "이득 ×1000": g["이득"].mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "rC model_13": (d["C 실제"] - d["C 17"]).mean(),
                 "rC model_14": (d["C 실제"] - d["C 18"]).mean(), "이득 ×1000": d["이득"].mean(),
                 "존 밖 AUC 17": auc(d["C 17"].to_numpy(), oc), "존 밖 AUC 18": auc(d["C 18"].to_numpy(), oc)})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(4))
gain = pd.Series(BST_L[fc.VAL_YEAR].feature_importance("gain"), index=LOC_FEAT)
print("변수 중요도 (gain 비율, 2019 모델):", (gain / gain.sum()).sort_values(ascending=False).round(3).to_dict())

공        rC model_13         rC model_14         이득 ×1000         존 밖 AUC 17         존 밖 AUC 18        
연도       2019   2020        2019    2020        2019    2020     2019    2020       2019    2020       2019    2020
변수 값                                                                                                               
아웃 0     4645   4375     -0.0324 -0.0260     -0.0202 -0.0058   9.2261  5.0698        NaN     NaN        NaN     NaN
   1     4500   4222     -0.0079 -0.0148     -0.0018 -0.0051   0.5837  3.6701        NaN     NaN        NaN     NaN
   2     4313   4018     -0.0106  0.0168     -0.0125  0.0190   1.0091  0.8212        NaN     NaN        NaN     NaN
전체      13458  12615     -0.0173 -0.0086     -0.0116  0.0023   3.7029  3.2481     0.5483  0.5428     0.5546  0.5524
주자 1루만   2886   2723     -0.0228  0.0026     -0.0299  0.0045  -0.3585 -0.1623        NaN     NaN        NaN     NaN
   득점권   3346   3136      0.0367  0.0266      0.0303  0.0159   3.7130  5.8314        NaN     NaN        NaN     NaN
   없음    7226   6756     -0.0400 -0.0295     -0.0236 -0.0049   5.3204  3.4235        NaN     NaN        NaN     NaN

변수 중요도 (gain 비율, 2019 모델): {'sdiff': 0.206, 'batorder': 0.194, 'inning': 0.142, 'out_cn': 0.123, 'base1': 0.076, 'FAST': 0.036, 'home': 0.035, 'base2': 0.031, 'same_side': 0.03, 'first_open': 0.02, 'CHUP': 0.02, 'base3': 0.019, 'CURV': 0.019, 'SLID': 0.017, 'TWOS': 0.015, 'FORK': 0.014, 'CUTT': 0.002, 'KNUC': 0.0, 'SINK': 0.0}


#### 🔍 3-3 결과: 상황이 존 밖 치우침을 줄임 (특히 주자 없음·0아웃)
- 전체 이득 ×1000 (실제 구종): 2019 +3.70, 2020 +3.25 (너클 영향 없음: 너클 빼도 +3.20)
- 존 밖 AUC: 2019 0.548 -> 0.555, 2020 0.543 -> 0.552
- 존 밖(C) 실제 − 예측 (2019 / 2020)
  - 주자 없음: −0.040 -> −0.024 / −0.030 -> −0.005 (이득 +5.3 / +3.4)
  - 득점권: +0.037 -> +0.030 / +0.027 -> +0.016 (이득 +3.7 / +5.8)
  - 1루만: 거의 그대로 (이득 −0.4 / −0.2)
  - 0아웃: −0.032 -> −0.020 / −0.026 -> −0.006 (이득 +9.2 / +5.1)
- 4-9·4-23에서 본 득점권 피해 가기 / 주자 없을 때 존 안으로가 일부 반영됨, 치우침이 완전히 사라지지는 않음 (규제가 강해 보정이 작음)
- 변수 중요도 (gain): 점수 차 0.21, 타순 0.19, 이닝 0.14, 아웃 0.12, 1루 주자 0.08 -> 주자보다 점수 차·타순·이닝 몫이 큼

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_12과 같음, 위치만 3-1에서 고른 값 (분산 따로 포함)

In [17]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
        P = loc_correct(P, q, year, t)  # 상황 보정 (model_14), 정규분포 g는 model_13 그대로
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [18]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종·좌표(정규분포)는 model_13과 같아야 함, 24칸만 바뀜
chk = pp.merge(m13, on=fc.KEYS, suffixes=("", "_13"))
same = ["ll_type", "nll_coord_true", "dist_true"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_13"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_13과 다른 열: {bad}"
print("구종·좌표 = model_13 (재현 확인), 24칸 확률만 바뀜")

(26073, 48)
구종·좌표 = model_13 (재현 확인), 24칸 확률만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_13과 나란히 / 경기 단위 부트스트랩 (model_14 − model_13, 2020)
- 2020은 확인용, 2020 너클 17구가 위치 숫자를 크게 흔들 수 있음 (4-27) -> 너클 뺀 값도 봄

In [19]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m13_sum = eval_years(m13)
display(pd.concat({MODEL: summary, "model_13": m13_sum, "차이 (model_13 대비)": summary - m13_sum}, axis=1).round(4))

model_14          model_13          차이 (model_13 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9429   2.9403   2.9471   2.9440          -0.0042 -0.0037
24칸 top1              0.1329   0.1328   0.1306   0.1325           0.0023  0.0002
24칸 top3              0.2951   0.2978   0.2893   0.2953           0.0058  0.0025
H/S/C logloss         1.0736   1.0744   1.0753   1.0759          -0.0017 -0.0015
H/S/C 정확도             0.4014   0.3983   0.3979   0.3915           0.0035  0.0067
24칸 logloss (실제 구종)   2.8835   2.8839   2.8872   2.8871          -0.0037 -0.0032
구종×칸 logloss          3.9448   3.9399   3.9485   3.9432          -0.0037 -0.0032
좌표 평균거리 cm           32.6377  31.8198  32.6377  31.8198           0.0000  0.0000
좌표 NLL                9.3124   9.2662   9.3124   9.2662           0.0000  0.0000

In [20]:
display(fc.bootstrap_diff(fc.pitch_scores(m13, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):  # 24칸 log loss 차이: 전체 / 너클 뺌
    a17, a18 = fc.align(m13, fp, yr), fc.align(pp, fp, yr)
    nk = a17["pitch_type"].ne("KNUC").to_numpy()
    d_true = a18["ll_zone_true"].to_numpy() - a17["ll_zone_true"].to_numpy()
    print(f"{yr} 24칸 실제 구종 차이 (model_14 − model_13): 전체 {d_true.mean():+.5f}, 너클 뺌 {d_true[nk].mean():+.5f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0037,-0.0059,-0.0016,좋아짐
ll_joint,-0.0032,-0.0055,-0.0010,좋아짐
nll_coord,0.0000,0.0000,0.0000,차이 불확실
dist,0.0000,0.0000,0.0000,차이 불확실


2019 24칸 실제 구종 차이 (model_14 − model_13): 전체 -0.00370, 너클 뺌 -0.00370
2020 24칸 실제 구종 차이 (model_14 − model_13): 전체 -0.00325, 너클 뺌 -0.00320


### 🥇1등🥇
#### 2020 24칸 log loss 실전 2.9440 -> 2.9403 (−0.0037), 실제 구종 −0.0032, 24칸·구종×칸 부트스트랩 좋아짐 -> 채택
- 구종·좌표(정규분포)는 model_13과 같음 (재현 확인) -> 24칸 확률만 상황으로 보정
- 2019 실제 구종 −0.0037, 실전 −0.0042 / 2020 실제 구종 −0.0032 (너클 빼도 −0.0032), 실전 −0.0037
- 부트스트랩 (2020, model_14 − model_13): 24칸 −0.0037 (95% −0.0059 ~ −0.0016), 구종×칸 −0.0032 (95% −0.0055 ~ −0.0010) 좋아짐
- 실전 개선이 실제 구종 기준보다 큼 (상황 효과는 구종과 상관없이 비슷하게 걸림) -> 3단계 위치 모델 중 실전 개선이 가장 큼 (model_11–13 실전 −0.0001 ~ −0.0008)
- H/S/C 정확도 +0.35%p / +0.67%p, 존 밖 AUC 0.548 -> 0.555 / 0.543 -> 0.552
- **채택 -> 다음 비교 출발점** (사용자 결정 2026-11-11)

## 6. 저장

- `results/model_14/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [21]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_14 ['per_pitch.parquet', 'summary.csv']
